# Import

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import math
import seaborn as sns
import pickle
import copy

import torch
from torch import nn, Tensor

import time
import joblib

from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [36]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import fm.preprocessing as preProcess

# Load the Datasets

In [37]:
# Inserire i nomi dei sample: 
sample_DY = "DYJetsToLL_M-50-madgraphMLM.npz"
sample_H = "GluGluHToTauTau_M125.npz"

In [38]:
data_DY = np.load(sample_DY)
data_H = np.load(sample_H)

## Check the content 

In [39]:
# checking the content:
print(data_DY.files)
print("\nFeatures and shapes - DY:")
for k in data_DY.files:
    print(k, data_DY[k].shape)

print("\n",data_H.files)
print("\nFeatures and shapes - Higgs:")
for k in data_H.files:
    print(k, data_H[k].shape)

['x_taus', 'x_tauprod', 'x_met', 'x_jets', 'x_neutrinos', 'x_gen', 'm_vis', 'm_vis_ptcorr', 'm_gen', 'm_gen_pt_vis_angles', 'm_vis_ptcorr_plus_nu', 'm_tauprod_nu', 'm_vis_nu_vis', 'm_fastmtt']

Features and shapes - DY:
x_taus (534715, 2, 18)
x_tauprod (534715, 10, 10)
x_met (534715, 1, 8)
x_jets (534715, 3, 4)
x_neutrinos (534715, 2, 1, 5)
x_gen (534715, 2, 4)
m_vis (534715,)
m_vis_ptcorr (534715,)
m_gen (534715,)
m_gen_pt_vis_angles (534715,)
m_vis_ptcorr_plus_nu (534715,)
m_tauprod_nu (534715,)
m_vis_nu_vis (534715,)
m_fastmtt (534715,)

 ['x_taus', 'x_tauprod', 'x_met', 'x_jets', 'x_neutrinos', 'x_gen', 'm_vis', 'm_vis_ptcorr', 'm_gen', 'm_gen_pt_vis_angles', 'm_vis_ptcorr_plus_nu', 'm_tauprod_nu', 'm_vis_nu_vis', 'm_fastmtt']

Features and shapes - Higgs:
x_taus (710060, 2, 18)
x_tauprod (710060, 10, 10)
x_met (710060, 1, 8)
x_jets (710060, 3, 4)
x_neutrinos (710060, 2, 1, 5)
x_gen (710060, 2, 4)
m_vis (710060,)
m_vis_ptcorr (710060,)
m_gen (710060,)
m_gen_pt_vis_angles (710060,)


## Preparing the columns for the DataFrames

In [40]:
x_tau_features = ['logpt', 'eta', 'phi', 'mass', 'dxy', 'dz', 'ptCorrPNet', 'rawPNetVSjet',
                  'rawDeepTau2018v2p5VSjet', 'charge', 'dM_0', 'dM_1', 'dM_2', 'dM_10',
                  'dM_11', 'leadTkDeltaEta', 'leadTkDeltaPhi', 'leadTkPtOverTauPt']
x_met_features = [
    'MET_logpt', 'MET_phi', 'MET_covXX', 'MET_covXY',
    'MET_covYY', 'MET_significance',
    'MET_sumEt', 'MET_sumPtUnclustered'
]

x_jets_features = ['logpt', 'eta', 'phi', 'mass']

x_gen_features = ['pt', 'eta', 'phi', 'mass']

In [41]:
# DY :

X_taus  = data_DY["x_taus"]      # (N, 2, 18)
X_met   = data_DY["x_met"]       # (N, 1, 8)
X_jets  = data_DY["x_jets"]      # (N, 3, 4)
X_gen   = data_DY["x_gen"]       # (N, 2, 4)

N = X_taus.shape[0]

tau_columns = []
for i in [1, 2]:
    for feat in x_tau_features:
        tau_columns.append(f"tau{i}_{feat}")

met_columns = x_met_features.copy()

jet_columns = []
for j in range(1,4):
    for feat in x_jets_features:
        jet_columns.append(f"jet{j}_{feat}")

gen_columns = []
for i in [1, 2]:
    for feat in x_gen_features:
        gen_columns.append(f"tau{i}_gen_{feat}")

X_taus_flat     = X_taus.reshape(N, -1)
X_met_flat      = X_met.reshape(N, -1)
X_jets_flat     = X_jets.reshape(N, -1)
X_gen_flat      = X_gen.reshape(N, -1)

X_all = np.concatenate(
    [X_taus_flat, X_met_flat, X_jets_flat, X_gen_flat],
    axis=1
)

all_columns = (
    tau_columns +
    met_columns +
    jet_columns +
    gen_columns
)

print(X_all.shape)
print(len(all_columns))

(534715, 64)
64


In [42]:
# Higgs :

X_taus_H  = data_H["x_taus"]
X_met_H   = data_H["x_met"]
X_jets_H  = data_H["x_jets"]
X_gen_H   = data_H["x_gen"]

N_H = X_taus_H.shape[0]

# ---- flatten
X_taus_flat_H = X_taus_H.reshape(N_H, -1)
X_met_flat_H  = X_met_H.reshape(N_H, -1)
X_jets_flat_H = X_jets_H.reshape(N_H, -1)
X_gen_flat_H  = X_gen_H.reshape(N_H, -1)

# ---- concat
X_all_H = np.concatenate(
    [X_taus_flat_H, X_met_flat_H, X_jets_flat_H, X_gen_flat_H],
    axis=1
)

print(X_all_H.shape)
print(len(all_columns))   # è lo stesso di DY

(710060, 64)
64


## Create the DataFrames

In [43]:
df_DY = pd.DataFrame(X_all, columns=all_columns)
df_DY["class"] = 0

In [44]:
df_DY

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,jet3_mass,tau1_gen_pt,tau1_gen_eta,tau1_gen_phi,tau1_gen_mass,tau2_gen_pt,tau2_gen_eta,tau2_gen_phi,tau2_gen_mass,class
0,3.663993,1.052002,-0.143311,3.996094,0.521484,0.239021,0.916992,0.776855,-1.000000,1.0,...,0.000000,39.625,1.082031,-0.107422,0.0,34.0000,0.004623,-2.960938,0.0,0
1,3.374451,0.901245,0.105301,2.816406,0.000015,0.005840,1.041992,0.998047,-1.000000,1.0,...,0.000000,35.375,0.908203,0.114502,0.0,24.1250,-1.179688,-2.367188,0.0,0
2,3.674162,0.875122,-0.090088,0.646484,-0.000050,0.000212,1.153320,0.914062,0.984863,-1.0,...,0.000000,46.625,0.875000,-0.092529,0.0,39.1250,1.570312,3.062500,0.0,0
3,3.752792,-2.368652,0.984741,0.701660,0.005585,8.266113,1.101562,0.840820,0.741699,-1.0,...,0.000000,46.375,-2.390625,0.986328,0.0,33.1250,-1.375000,-2.304688,0.0,0
4,3.473854,1.000977,1.160889,1.307617,-0.002544,-0.000709,1.008789,0.070068,0.581543,-1.0,...,0.000000,39.500,0.992188,1.148438,0.0,40.5000,2.023438,-1.933594,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534710,3.144562,1.772461,2.792480,4.429688,0.004097,-0.026582,1.059570,0.164429,-1.000000,1.0,...,5.992188,26.000,1.765625,2.789062,0.0,42.6250,-0.165527,-1.417969,0.0,0
534711,3.289238,-1.037598,0.081070,0.643066,0.000596,-0.001515,1.074219,0.555664,0.969727,-1.0,...,3.630859,58.625,-1.042969,0.100342,0.0,33.7500,-1.523438,-3.085938,0.0,0
534712,3.719521,-0.274231,-1.346191,0.774902,-0.008469,-0.004040,1.018555,0.997559,0.997559,1.0,...,3.896484,42.875,-0.275391,-1.347656,0.0,42.1250,-0.994141,1.722656,0.0,0
534713,3.798122,0.109604,2.325195,1.334961,0.000878,0.003865,1.055664,0.676270,0.972656,-1.0,...,0.000000,48.625,0.106934,2.328125,0.0,42.5000,0.429688,-0.990234,0.0,0


In [45]:
df_H = pd.DataFrame(X_all_H, columns=all_columns)
df_H["class"] = 1

In [46]:
df_H

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,jet3_mass,tau1_gen_pt,tau1_gen_eta,tau1_gen_phi,tau1_gen_mass,tau2_gen_pt,tau2_gen_eta,tau2_gen_phi,tau2_gen_mass,class
0,3.750918,0.149750,-2.504883,0.919922,0.001779,0.000654,1.097656,0.985840,0.997559,1.0,...,0.000000,55.875,0.148438,-2.492188,0.0,45.125,-1.218750,0.826172,0.0,1
1,3.908546,-0.357422,-2.585938,0.139526,0.006088,0.012403,0.971680,0.966797,0.997559,1.0,...,0.000000,62.000,-0.367188,-2.601562,0.0,46.500,-1.496094,0.503906,0.0,1
2,4.752102,-0.059525,-1.178711,1.112305,0.000178,-0.007251,1.041016,0.557617,0.974609,1.0,...,5.726562,133.000,-0.059448,-1.175781,0.0,48.125,-0.636719,-2.789062,0.0,1
3,3.857435,-1.089844,0.347168,0.917480,0.001588,-4.294922,1.061523,0.937500,0.985840,-1.0,...,3.828125,57.875,-1.089844,0.329102,0.0,66.500,-1.351562,-2.898438,0.0,1
4,3.732360,1.662109,-2.540039,0.139526,0.001172,-0.001545,1.010742,0.907715,0.957031,1.0,...,2.125000,57.250,1.679688,-2.539062,0.0,38.375,0.092773,0.521484,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710055,3.353004,1.791992,-0.711182,0.311279,0.000830,0.000770,1.134766,0.394043,0.836914,1.0,...,3.943359,36.000,1.792969,-0.714844,0.0,27.625,-0.822266,2.578125,0.0,1
710056,4.464565,-0.439941,3.046875,1.164062,-0.001043,-0.000269,1.002930,0.691406,0.961426,-1.0,...,5.289062,127.750,-0.436523,3.046875,0.0,38.750,0.552734,-1.625000,0.0,1
710057,3.419531,1.242676,1.503174,0.644531,-0.000745,-0.000320,1.336914,0.785156,0.925781,-1.0,...,0.000000,53.000,1.250000,1.511719,0.0,66.750,1.882812,-1.667969,0.0,1
710058,4.078551,-0.174530,-2.548828,0.606934,0.002081,-0.000342,1.098633,0.931152,0.975586,-1.0,...,0.000000,95.750,-0.160645,-2.554688,0.0,36.125,0.691406,0.033691,0.0,1


In [47]:
# To 'match' the pt reco :

df_DY["tau1_gen_logpt"] = np.log(df_DY["tau1_gen_pt"])
df_DY["tau2_gen_logpt"] = np.log(df_DY["tau2_gen_pt"])

df_H["tau1_gen_logpt"] = np.log(df_H["tau1_gen_pt"])
df_H["tau2_gen_logpt"] = np.log(df_H["tau2_gen_pt"])

In [48]:
# sanity check
df_DY["tau1_gen_logpt"]

0         3.679460
1         3.566005
2         3.842137
3         3.836761
4         3.676301
            ...   
534710    3.258097
534711    4.071161
534712    3.758289
534713    3.884138
534714    3.548180
Name: tau1_gen_logpt, Length: 534715, dtype: float64

In [49]:
# altro check sulle correzioni PNet :
n_zeros = (df_DY["tau1_ptCorrPNet"] == 0).sum()
print("Numero di zeri:", n_zeros)

n_zeros_H = (df_H["tau1_ptCorrPNet"] == 0).sum()
print("Numero di zeri:", n_zeros_H)

n_zeros_2 = (df_DY["tau2_ptCorrPNet"] == 0).sum()
print("Numero di zeri:", n_zeros_2)

n_zeros_H_2 = (df_H["tau2_ptCorrPNet"] == 0).sum()
print("Numero di zeri:", n_zeros_H_2)

Numero di zeri: 39
Numero di zeri: 19
Numero di zeri: 126
Numero di zeri: 113


In [50]:
# Aggiungo altre colonne che mi servono :
for df in [df_DY, df_H]:

    # pt reco (non corretto)
    df["tau1_pt_reco"] = np.exp(df["tau1_logpt"])
    df["tau2_pt_reco"] = np.exp(df["tau2_logpt"])

    # pt reco corretto con PNet
    df["tau1_pt_reco_corrPNet"] = df["tau1_pt_reco"] * df["tau1_ptCorrPNet"]
    df["tau2_pt_reco_corrPNet"] = df["tau2_pt_reco"] * df["tau2_ptCorrPNet"]

In [51]:
# Aggiungo le masse, mi possono servire per confronti
df_DY["m_fastmtt"] = data_DY["m_fastmtt"]
df_DY["m_gen"]     = data_DY["m_gen"]
df_DY["m_vis"]     = data_DY["m_vis"]

In [52]:
eps = 1e-8

for df in [df_DY, df_H]:
    df["tau1_corr"] = df["tau1_gen_pt"] / np.clip(df["tau1_pt_reco_corrPNet"], eps, None)
    df["tau2_corr"] = df["tau2_gen_pt"] / np.clip(df["tau2_pt_reco_corrPNet"], eps, None)

y_cols = ["tau1_corr", "tau2_corr"]

In [53]:
# Ho alcune colonne con ptCorrPNet == 0, rimuovo questi casi
for df in [df_DY, df_H]:
    mask = (df["tau1_pt_reco_corrPNet"] > 0) & (df["tau2_pt_reco_corrPNet"] > 0)
    df.drop(df.index[~mask], inplace=True)

In [54]:
df_DY

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,tau2_gen_logpt,tau1_pt_reco,tau2_pt_reco,tau1_pt_reco_corrPNet,tau2_pt_reco_corrPNet,m_fastmtt,m_gen,m_vis,tau1_corr,tau2_corr
0,3.663993,1.052002,-0.143311,3.996094,0.521484,0.239021,0.916992,0.776855,-1.000000,1.0,...,3.526361,39.016822,21.071611,35.778121,19.137303,83.349709,83.660751,64.352629,1.107520,1.776635
1,3.374451,0.901245,0.105301,2.816406,0.000015,0.005840,1.041992,0.998047,-1.000000,1.0,...,3.183249,29.208235,22.322851,30.434752,24.742613,107.766106,91.321297,79.461255,1.162323,0.975038
2,3.674162,0.875122,-0.090088,0.646484,-0.000050,0.000212,1.153320,0.914062,0.984863,-1.0,...,3.666762,39.415600,35.794425,45.458812,35.007927,108.180054,90.633972,79.792117,1.025654,1.117604
3,3.752792,-2.368652,0.984741,0.701660,0.005585,8.266113,1.101562,0.840820,0.741699,-1.0,...,3.500288,42.639952,20.428112,46.970572,21.305882,86.091255,88.521049,67.317516,0.987320,1.554735
4,3.473854,1.000977,1.160889,1.307617,-0.002544,-0.000709,1.008789,0.070068,0.581543,-1.0,...,3.701302,32.260831,23.807997,32.544373,25.877247,78.009315,90.834152,63.348499,1.213727,1.565081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534710,3.144562,1.772461,2.792480,4.429688,0.004097,-0.026582,1.059570,0.164429,-1.000000,1.0,...,3.752441,23.209502,21.456071,24.592099,16.972087,84.256409,94.222244,65.006072,1.057250,2.511477
534711,3.289238,-1.037598,0.081070,0.643066,0.000596,-0.001515,1.074219,0.555664,0.969727,-1.0,...,3.518980,26.822429,23.019591,28.813157,21.715747,69.251572,91.520706,51.524930,2.034661,1.554172
534712,3.719521,-0.274231,-1.346191,0.774902,-0.008469,-0.004040,1.018555,0.997559,0.997559,1.0,...,3.740641,41.244645,26.120728,42.009926,24.488182,94.836197,90.494019,69.658072,1.020592,1.720218
534713,3.798122,0.109604,2.325195,1.334961,0.000878,0.003865,1.055664,0.676270,0.972656,-1.0,...,3.749504,44.617322,31.909392,47.100903,33.405145,104.234131,91.755081,76.215333,1.032358,1.272259


In [55]:
df_H

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,tau2_gen_mass,class,tau1_gen_logpt,tau2_gen_logpt,tau1_pt_reco,tau2_pt_reco,tau1_pt_reco_corrPNet,tau2_pt_reco_corrPNet,tau1_corr,tau2_corr
0,3.750918,0.149750,-2.504883,0.919922,0.001779,0.000654,1.097656,0.985840,0.997559,1.0,...,0.0,1,4.023117,3.809436,42.560151,38.164637,46.716416,44.649644,1.196046,1.010646
1,3.908546,-0.357422,-2.585938,0.139526,0.006088,0.012403,0.971680,0.966797,0.997559,1.0,...,0.0,1,4.127134,3.839452,49.826450,41.729680,48.415350,42.992980,1.280586,1.081572
2,4.752102,-0.059525,-1.178711,1.112305,0.000178,-0.007251,1.041016,0.557617,0.974609,1.0,...,0.0,1,4.890349,3.873802,115.827486,30.399418,120.578223,28.588515,1.103018,1.683368
3,3.857435,-1.089844,0.347168,0.917480,0.001588,-4.294922,1.061523,0.937500,0.985840,-1.0,...,0.0,1,4.058286,4.197202,47.343736,22.447759,50.256485,16.463152,1.151593,4.039324
4,3.732360,1.662109,-2.540039,0.139526,0.001172,-0.001545,1.010742,0.907715,0.957031,1.0,...,0.0,1,4.047428,3.647406,41.777582,21.636590,42.226365,19.682113,1.355788,1.949740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710055,3.353004,1.791992,-0.711182,0.311279,0.000830,0.000770,1.134766,0.394043,0.836914,1.0,...,0.0,1,3.583519,3.318721,28.588498,23.995189,32.441244,22.952429,1.109698,1.203576
710056,4.464565,-0.439941,3.046875,1.164062,-0.001043,-0.000269,1.002930,0.691406,0.961426,-1.0,...,0.0,1,4.850075,3.657131,86.883251,24.465774,87.137792,21.837615,1.466069,1.774461
710057,3.419531,1.242676,1.503174,0.644531,-0.000745,-0.000320,1.336914,0.785156,0.925781,-1.0,...,0.0,1,3.970292,4.200954,30.555070,25.188037,40.849503,32.247574,1.297445,2.069923
710058,4.078551,-0.174530,-2.548828,0.606934,0.002081,-0.000342,1.098633,0.931152,0.975586,-1.0,...,0.0,1,4.561741,3.586985,59.059847,28.813764,64.885086,31.768301,1.475686,1.137140


# Pre-Processing

## pT cut

In [56]:
pt_cfg = preProcess.PtCutConfig(
    pt_min=30.0,
    space="pt",
    tau1_pt_col="tau1_pt_reco",
    tau2_pt_col="tau2_pt_reco",
)

In [57]:
df_DY_pt = preProcess.apply_pt_cut(df_DY, pt_cfg)
df_H_pt  = preProcess.apply_pt_cut(df_H,  pt_cfg)

In [58]:
print("DY:", len(df_DY), "->", len(df_DY_pt))
print("H :", len(df_H),  "->", len(df_H_pt))

DY: 534550 -> 109991
H : 709928 -> 316472


In [27]:
pt_min = 30.0

print("DY min tau1_pt_reco:", df_DY_pt["tau1_pt_reco"].min())
print("DY min tau2_pt_reco:", df_DY_pt["tau2_pt_reco"].min())

print("H  min tau1_pt_reco:", df_H_pt["tau1_pt_reco"].min())
print("H  min tau2_pt_reco:", df_H_pt["tau2_pt_reco"].min())

# Tolleranza numerica piccola
assert df_DY_pt["tau1_pt_reco"].min() > pt_min - 1e-12
assert df_DY_pt["tau2_pt_reco"].min() > pt_min - 1e-12
assert df_H_pt["tau1_pt_reco"].min()  > pt_min - 1e-12
assert df_H_pt["tau2_pt_reco"].min()  > pt_min - 1e-12

print("pT reco corrPNet cut OK (pt > 30 GeV)")

DY min tau1_pt_reco: 30.020536419815844
DY min tau2_pt_reco: 30.000008706844365
H  min tau1_pt_reco: 30.066830700924445
H  min tau2_pt_reco: 30.000015859404666
pT reco corrPNet cut OK (pt > 30 GeV)


In [32]:
# check su ptrecoCorrPNet:

print(df_DY_pt["tau1_pt_reco_corrPNet"].min())
print(df_DY_pt["tau2_pt_reco_corrPNet"].min())
print(df_H_pt["tau1_pt_reco_corrPNet"].min())
print(df_H_pt["tau2_pt_reco_corrPNet"].min())

17.683272213254597
17.138952728757577
18.89620655398476
16.860135413626903


## $\tau$ ID

In [64]:
# valid taus:
tauid_cfg = preProcess.TauIDConfig(
    tau1_id_col="tau1_rawPNetVSjet",
    tau2_id_col="tau2_rawPNetVSjet",
    invalid_value=-1.0,
)

df_DY_valid = preProcess.filter_valid_tauid(df_DY_pt, tauid_cfg)
df_H_valid  = preProcess.filter_valid_tauid(df_H_pt,  tauid_cfg)

In [65]:
# check
print("DY after pt:", len(df_DY_pt), "-> after valid tauID:", len(df_DY_valid))
print("H  after pt:", len(df_H_pt),  "-> after valid tauID:", len(df_H_valid))

assert (df_DY_valid["tau1_rawPNetVSjet"] != -1).all()
assert (df_DY_valid["tau2_rawPNetVSjet"] != -1).all()
assert (df_H_valid["tau1_rawPNetVSjet"]  != -1).all()
assert (df_H_valid["tau2_rawPNetVSjet"]  != -1).all()

print("valid tauID OK (no -1)")

DY after pt: 109991 -> after valid tauID: 109321
H  after pt: 316472 -> after valid tauID: 314955
valid tauID OK (no -1)


In [67]:
# calcolo sulle soglie:
tauid_cfg = preProcess.TauIDConfig(
    tau1_id_col="tau1_rawPNetVSjet",
    tau2_id_col="tau2_rawPNetVSjet",
    invalid_value=-1.0,
    wp_mode="target_eff",
    target_eff=0.90,
    reference="higgs",
    require_both=True,
)

In [68]:
# soglie:
thr1, thr2 = preProcess.compute_tauid_thresholds(df_DY_valid, df_H_valid, tauid_cfg)
print("thr_tau1 =", thr1)
print("thr_tau2 =", thr2)

thr_tau1 = 0.6416015625
thr_tau2 = 0.493408203125


In [69]:
# applico WP:
df_DY_tauid = preProcess.apply_tauid_wp(df_DY_valid, tauid_cfg, thr1, thr2)
df_H_tauid  = preProcess.apply_tauid_wp(df_H_valid,  tauid_cfg, thr1, thr2)

In [70]:
# check:
print("DY after valid:", len(df_DY_valid), "-> after WP:", len(df_DY_tauid))
print("H  after valid:", len(df_H_valid),  "-> after WP:", len(df_H_tauid))

assert (df_DY_tauid["tau1_rawPNetVSjet"] > thr1).all()
assert (df_DY_tauid["tau2_rawPNetVSjet"] > thr2).all()
assert (df_H_tauid["tau1_rawPNetVSjet"]  > thr1).all()
assert (df_H_tauid["tau2_rawPNetVSjet"]  > thr2).all()

print("tauID WP OK")

DY after valid: 109321 -> after WP: 81947
H  after valid: 314955 -> after WP: 255404
tauID WP OK


# Save the valid dataframes

In [74]:
df_DY_event = df_DY_tauid.copy()
df_H_event  = df_H_tauid.copy()

In [75]:
# check rapido
print("Final DY events:", len(df_DY_event))
print("Final H events :", len(df_H_event))

Final DY events: 81947
Final H events : 255404


In [76]:
df_DY_event.to_pickle("df_DY_event.pkl")
df_H_event.to_pickle("df_H_event.pkl")

In [77]:
df_H_event

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,tau2_gen_mass,class,tau1_gen_logpt,tau2_gen_logpt,tau1_pt_reco,tau2_pt_reco,tau1_pt_reco_corrPNet,tau2_pt_reco_corrPNet,tau1_corr,tau2_corr
0,3.750918,0.149750,-2.504883,0.919922,0.001779,0.000654,1.097656,0.985840,0.997559,1.0,...,0.0,1,4.023117,3.809436,42.560151,38.164637,46.716416,44.649644,1.196046,1.010646
1,3.908546,-0.357422,-2.585938,0.139526,0.006088,0.012403,0.971680,0.966797,0.997559,1.0,...,0.0,1,4.127134,3.839452,49.826450,41.729680,48.415350,42.992980,1.280586,1.081572
6,3.735467,0.549683,1.355957,0.788574,0.002239,0.000944,1.020508,0.999023,0.999512,-1.0,...,0.0,1,4.090169,3.634291,41.907610,31.002881,42.767043,31.457024,1.397104,1.204024
10,3.729789,0.430664,-0.860352,0.687012,0.011436,-0.025360,1.043945,0.988281,0.999512,1.0,...,0.0,1,3.988984,4.248495,41.670316,37.238447,43.501531,38.147589,1.241336,1.834978
14,4.210932,1.539795,0.115631,8.835938,-0.506348,1.439453,0.895996,0.809082,-1.000000,1.0,...,0.0,1,4.215824,4.114964,67.419363,35.936132,60.407485,38.603266,1.121550,1.586653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710044,3.951714,-1.898438,2.384766,1.361328,0.000673,0.003083,0.961426,0.984863,0.989258,-1.0,...,0.0,1,3.991296,4.174387,52.024475,36.990902,50.017671,43.962820,1.082118,1.478522
710046,4.037437,0.069366,2.712402,0.793945,0.000558,0.003005,1.001953,0.969238,0.924805,-1.0,...,0.0,1,4.158883,4.011868,56.680881,36.487017,56.791586,39.444460,1.126927,1.400704
710048,4.366798,-1.340820,2.247559,0.798340,-0.002230,0.004375,1.019531,0.998535,0.991211,1.0,...,0.0,1,4.607667,4.564348,78.790970,50.086421,80.329856,51.994009,1.247979,1.846367
710049,4.094778,0.109863,-2.218750,0.991699,-0.003817,-0.010967,1.064453,0.997559,0.999023,1.0,...,0.0,1,4.297285,3.881564,60.026016,45.053121,63.894880,44.965126,1.150327,1.078614


In [78]:
df_DY_event

,tau1_logpt,tau1_eta,tau1_phi,tau1_mass,tau1_dxy,tau1_dz,tau1_ptCorrPNet,tau1_rawPNetVSjet,tau1_rawDeepTau2018v2p5VSjet,tau1_charge,...,tau2_gen_logpt,tau1_pt_reco,tau2_pt_reco,tau1_pt_reco_corrPNet,tau2_pt_reco_corrPNet,m_fastmtt,m_gen,m_vis,tau1_corr,tau2_corr
2,3.674162,0.875122,-0.090088,0.646484,-0.000050,0.000212,1.153320,0.914062,0.984863,-1.0,...,3.666762,39.415600,35.794425,45.458812,35.007927,108.180054,90.633972,79.792117,1.025654,1.117604
6,3.694579,2.389160,1.977051,1.098633,-0.001568,0.005698,1.026367,0.996094,0.994629,1.0,...,3.817712,40.228638,38.692733,41.289354,41.035457,108.058052,91.356873,79.008902,1.108034,1.108797
9,3.875156,-0.371704,0.386658,1.157227,0.000169,-0.001055,1.019531,0.999023,0.997559,-1.0,...,3.448001,48.190200,30.784226,49.131415,30.844352,115.543953,92.915504,88.303589,1.076195,1.019230
17,3.964924,0.317871,1.018311,1.087891,-0.001331,-0.000872,0.998535,0.888672,0.994629,1.0,...,3.912023,52.716280,30.928882,52.639059,31.170514,107.876442,106.433792,80.907332,1.075722,1.604080
25,3.511394,1.427734,0.866943,1.159180,0.000447,0.009053,0.967773,0.906738,0.992188,1.0,...,3.764103,33.494920,31.700107,32.415494,34.021892,96.287704,92.573875,69.744653,1.345807,1.267566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534692,3.549872,2.192871,-1.546631,0.139526,0.005932,-0.057304,1.031250,0.977051,0.975098,-1.0,...,3.772761,34.808876,33.687892,35.896653,38.688438,93.134178,91.583122,68.094399,1.354583,1.124367
534695,3.965171,2.100586,1.186768,5.187500,-0.154419,-8.217773,1.008789,0.995117,-1.000000,-1.0,...,3.537330,52.729291,30.729837,53.192732,31.930221,101.321236,91.403084,79.247046,1.189072,1.076566
534697,3.782284,1.195312,2.006836,1.136719,0.001639,-0.027286,0.903809,0.762695,0.982910,1.0,...,3.695110,43.916233,40.489248,39.691869,40.449708,122.874733,88.948853,90.031736,1.089644,0.995063
534706,3.714146,0.386902,-1.045898,0.139526,-0.001197,-0.002113,1.022461,0.974121,0.996582,-1.0,...,3.778492,41.023534,36.684761,41.944961,37.329610,105.329155,91.009682,77.764652,1.123496,1.171992
